# Initialization and Configuration


In [ ]:
import json

import boto3
import requests
from botocore import UNSIGNED
from botocore.config import Config
from botocore.exceptions import ClientError

# Configuration
MODEL_ID = "us.anthropic.claude-sonnet-4-20250514-v1:0"
SECRET_ID = "bedrock-gateway-dev-oauth-credentials"
API_URL = "https://your-gateway-url.com"
AWS_REGION = "us-east-1"

# Extract the secret value from Secret Manager
JSON_SECRET = json.loads(boto3.client('secretsmanager', region_name=AWS_REGION).get_secret_value(
    SecretId=SECRET_ID)['SecretString'])
CLIENT_ID = JSON_SECRET['client_id']
CLIENT_SECRET = JSON_SECRET['client_secret']
TOKEN_URL = JSON_SECRET['token_url']

# Token Generation

In [ ]:
def generate_token():
    """Generate access token for API authentication"""
    payload = {
        "client_id": CLIENT_ID,
        "client_secret": CLIENT_SECRET,
        "grant_type": "client_credentials",
        "audience": "bedrockproxygateway",
        "scope": "bedrockproxygateway:invoke"
    }

    response = requests.post(TOKEN_URL, data=payload)
    response.raise_for_status()
    token = response.json()['access_token']
    return f"Bearer {token}"

def create_bedrock_client(token):
    """Create and configure Bedrock runtime client"""
    bedrock = boto3.client(
        'bedrock-runtime',
        region_name=AWS_REGION,
        endpoint_url=API_URL,
        config=Config(signature_version=UNSIGNED, retries={'max_attempts': 0}),
        verify=False
    )

    def add_api_token(request, **kwargs):
        request.headers.add_header('Authorization', token)
        return request
    
    bedrock.meta.events.register('before-sign.bedrock-runtime.*', add_api_token)
    return bedrock

# Bedrock Runtime Client Setup


In [ ]:
# Generate token
TOKEN = generate_token()
print(f"Token generated: {TOKEN}")

# Create Bedrock client
bedrock = create_bedrock_client(TOKEN)
print("Bedrock client configured successfully")

## Converse (Bedrock)


In [ ]:
# Start a conversation with the user message.
user_message = "AWS Reinvent give me detail of 10 page "

conversation = [
    {
        "role": "user",
        "content": [{"text": user_message}],
    }
]

try:
    # Send the message to the model, using a basic inference configuration.
    response = bedrock.converse(
        modelId=MODEL_ID,
        messages=conversation,
        inferenceConfig={"maxTokens": 512, "temperature": 0.5},
        guardrailConfig={ "guardrailIdentifier": "baseline-security","guardrailVersion": "1"},
    )

    # Print response headers
    print("\nResponse Headers:")
    print("----------------")
    for key, value in response.get('ResponseMetadata', {}).get('HTTPHeaders', {}).items():
        print(f"{key}: {value}")

    print(f"Bedrock response time => {response['metrics']['latencyMs']}")

    # Extract and print the response text.
    response_text = response["output"]["message"]["content"][0]["text"]
    print(f"Bedrock response: {response_text}")
    
except ClientError as e:
    error_code = e.response.get('Error', {}).get('Code', 'Unknown')
    error_message = e.response.get('Error', {}).get('Message', str(e))
    status_code = e.response.get('ResponseMetadata', {}).get('HTTPStatusCode', 'Unknown')
    print(f"ERROR: Can't invoke '{MODEL_ID}'.")
    print(f"Error Code: {error_code}")
    print(f"Error Message: {error_message}")
    print(f"HTTP Status Code: {status_code}")
    
except Exception as e:
    print(f"ERROR: Unexpected error with '{MODEL_ID}'.\nReason: {e}")


## Converse Stream (Bedrock)


In [ ]:
# Start a conversation with the user message.
user_message = "AWS Reinvent give me detail of 10 page "
conversation = [
    {
        "role": "user",
        "content": [{"text": user_message}],
    }
]

try:
    # Send the message to the model, using a basic inference configuration.
    streaming_response = bedrock.converse_stream(
        modelId=MODEL_ID,
        messages=conversation,
        inferenceConfig={"maxTokens": 512},
    )

    # Extract and print the streamed response text in real-time.
    for chunk in streaming_response["stream"]:
        if "contentBlockDelta" in chunk:
            text = chunk["contentBlockDelta"]["delta"]["text"]
            print(text, end="")

except ClientError as e:
    error_code = e.response.get('Error', {}).get('Code', 'Unknown')
    error_message = e.response.get('Error', {}).get('Message', str(e))
    print(f"\nERROR: Can't invoke '{MODEL_ID}'.")
    print(f"Error Code: {error_code}")
    print(f"Error Message: {error_message}")
except Exception as e:
    print(f"\nERROR: Unexpected error. Reason: {e}")

## InvokeModel (Bedrock)


In [ ]:
# Define the prompt for the model.
prompt = "Describe the purpose of a 'hello world' program in one line."

# Format the request payload using the model's native structure.
native_request = {
    "anthropic_version": "bedrock-2023-05-31",
    "max_tokens": 512,
    "temperature": 0.5,
    "messages": [
        {
            "role": "user",
            "content": [{"type": "text", "text": prompt}],
        }
    ],
}

# Convert the native request to JSON.
request = json.dumps(native_request)

try:
    # Invoke the model with the request.
    response = bedrock.invoke_model(modelId=MODEL_ID, body=request)
    
    # Decode the response body.
    model_response = json.loads(response["body"].read())
    response_text = model_response["content"][0]["text"]
    print(response_text)

except ClientError as e:
    error_code = e.response.get('Error', {}).get('Code', 'Unknown')
    error_message = e.response.get('Error', {}).get('Message', str(e))
    print(f"ERROR: Can't invoke '{MODEL_ID}'.")
    print(f"Error Code: {error_code}")
    print(f"Error Message: {error_message}")
except Exception as e:
    print(f"ERROR: Unexpected error. Reason: {e}")

## InvokeModelWithResponseStream (Bedrock)


In [ ]:
# Define the prompt for the model.
prompt = "Describe the purpose of a 'hello world' program in one line."

# Format the request payload using the model's native structure.
native_request = {
    "anthropic_version": "bedrock-2023-05-31",
    "max_tokens": 512,
    "temperature": 0.5,
    "messages": [
        {
            "role": "user",
            "content": [{"type": "text", "text": prompt}],
        }
    ],
}

# Convert the native request to JSON.
request = json.dumps(native_request)

try:
    # Invoke the model with the request.
    streaming_response = bedrock.invoke_model_with_response_stream(
        modelId=MODEL_ID, body=request
    )
    
    # Extract and print the response text in real-time.
    for event in streaming_response["body"]:
        chunk = json.loads(event["chunk"]["bytes"])
        if chunk["type"] == "content_block_delta":
            print(chunk["delta"].get("text", ""), end="")

except ClientError as e:
    error_code = e.response.get('Error', {}).get('Code', 'Unknown')
    error_message = e.response.get('Error', {}).get('Message', str(e))
    print(f"\nERROR: Can't invoke '{MODEL_ID}'.")
    print(f"Error Code: {error_code}")
    print(f"Error Message: {error_message}")
except Exception as e:
    print(f"\nERROR: Unexpected error. Reason: {e}")

# Embedding Models


## Amazon Titan Text Embeddings


In [ ]:
# Amazon Titan Text Embeddings v2
TITAN_EMBEDDING_MODEL = "amazon.titan-embed-text-v1"

# Text to embed
text_to_embed = "AWS Bedrock is a fully managed service that offers foundation models from leading AI companies."

# Prepare request for Titan embedding model
titan_request = {
    "inputText": text_to_embed,
    "dimensions": 1024,  # Optional: 256, 512, 1024 (default)
    "normalize": True    # Optional: normalize embeddings
}

try:
    # Call Bedrock invoke model API for embeddings
    response = bedrock.invoke_model(
        modelId=TITAN_EMBEDDING_MODEL,
        body=json.dumps(titan_request),
        contentType="application/json",
        accept="application/json"
    )
    
    # Parse response
    response_body = json.loads(response["body"].read())
    embeddings = response_body["embedding"]
    
    print(f"Titan Embedding Model: {TITAN_EMBEDDING_MODEL}")
    print(f"Input text: {text_to_embed}")
    print(f"Embedding dimensions: {len(embeddings)}")
    print(f"First 5 embedding values: {embeddings[:5]}")
    print(f"Input token count: {response_body.get('inputTextTokenCount', 'N/A')}")
    
except ClientError as e:
    error_code = e.response.get('Error', {}).get('Code', 'Unknown')
    error_message = e.response.get('Error', {}).get('Message', str(e))
    status_code = e.response.get('ResponseMetadata', {}).get('HTTPStatusCode', 'Unknown')
    print(f"ERROR: Can't invoke '{TITAN_EMBEDDING_MODEL}'.")
    print(f"Status Code: {status_code}")
    print(f"Error Code: {error_code}")
    print(f"Error Message: {error_message}")
except Exception as e:
    print(f"ERROR: Unexpected error. Reason: {e}")

## Cohere Embeddings


In [ ]:
# Cohere Embed English v3 model
COHERE_EMBEDDING_MODEL = "cohere.embed-english-v3"

# Text to embed
text_to_embed = "Cohere provides powerful language models for enterprise applications."

# Prepare request for Cohere embedding model
cohere_request = {
    "texts": [text_to_embed],
    "input_type": "search_document"  # Options: search_document, search_query, classification, clustering
}

try:
    # Call Bedrock invoke model API for embeddings
    response = bedrock.invoke_model(
        modelId=COHERE_EMBEDDING_MODEL,
        body=json.dumps(cohere_request),
        contentType="application/json",
        accept="application/json"
    )
    
    # Parse response
    response_body = json.loads(response["body"].read())
    embeddings = response_body["embeddings"][0]
    
    print(f"Cohere Embedding Model: {COHERE_EMBEDDING_MODEL}")
    print(f"Input text: {text_to_embed}")
    print(f"Embedding dimensions: {len(embeddings)}")
    print(f"First 5 embedding values: {embeddings[:5]}")
    print(f"Response ID: {response_body.get('id', 'N/A')}")
    
except ClientError as e:
    error_code = e.response.get('Error', {}).get('Code', 'Unknown')
    error_message = e.response.get('Error', {}).get('Message', str(e))
    status_code = e.response.get('ResponseMetadata', {}).get('HTTPStatusCode', 'Unknown')
    print(f"ERROR: Can't invoke '{COHERE_EMBEDDING_MODEL}'.")
    print(f"Status Code: {status_code}")
    print(f"Error Code: {error_code}")
    print(f"Error Message: {error_message}")
except Exception as e:
    print(f"ERROR: Unexpected error. Reason: {e}")